# VoiceAI — train a Setswana voice (Piper / VITS)

**Run in Google Colab with a GPU** (Runtime -> Change runtime type -> GPU).

This fine-tunes a Piper TTS voice on real Setswana speech (NCHLT). Piper is
**MIT-licensed** (commercial use OK), runs inference on **CPU**, and exports a
small **~60 MB** voice you can drop straight into the website.

**Honest expectations:** the first run will be rough. A clean voice needs ~1-3h
of ONE speaker; our NCHLT data is many speakers with a little each, so we pick
the best-covered speaker. Treat run #1 as *proof the pipeline works and makes
Setswana sounds*, then iterate (more data, more steps, better speaker).

Checkpoints save to your Google Drive so a disconnect doesn't lose training.


## 1. GPU + Google Drive (so training survives disconnects)

In [ ]:
!nvidia-smi -L  # should show a GPU. If not: Runtime -> Change runtime type -> GPU.
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/voiceai_tts'
os.makedirs(WORK, exist_ok=True)
print('Working dir on Drive:', WORK)


## 2. Install Piper training + espeak-ng (Setswana phonemes)

In [ ]:
# espeak-ng provides Setswana (tn) phonemization that Piper needs.
!apt-get -qq install -y espeak-ng >/dev/null
# Piper training toolkit. Errors are NOT hidden — if pip prints red, tell Claude.
!git clone -q https://github.com/rhasspy/piper /content/piper
%cd /content/piper/src/python
!pip -q install -e .
# The official piper-phonemize has NO Linux wheel for Colab's Python 3.12;
# piper-phonemize-cross ships that wheel (same 'piper_phonemize' import name).
!pip install piper-phonemize-cross "pytorch-lightning==1.9.5" "torchmetrics==0.11.4" onnxruntime "numpy<2"
!bash build_monotonic_align.sh
print('Verify both import cleanly:')
import pytorch_lightning as pl; print('pytorch_lightning', pl.__version__)
import piper_phonemize; print('piper_phonemize OK')

## 3. Pull Setswana speech + pick the best-covered speaker

Streams NCHLT Setswana (`Max5ive/nchlt_speech_setswana` — a mirror with an
explicit `speaker_id` + `text` field), keeps the single speaker with the most
audio, resamples to 22.05 kHz mono, and writes an LJSpeech-style dataset. Raise
`MAX_CLIPS` for more data (slower).

In [ ]:
import os, io, soundfile as sf, librosa, numpy as np
from collections import defaultdict
from datasets import load_dataset, Audio

MAX_CLIPS = 4000  # scanned; we keep only the top speaker's clips
DATA = '/content/tsn_data'
os.makedirs(f'{DATA}/wavs', exist_ok=True)

# danielshaps/nchlt_speech_tsn was 503-ing; this mirror is the same NCHLT
# Setswana speech with explicit speaker_id + text fields.
ds = load_dataset('Max5ive/nchlt_speech_setswana', split='train', streaming=True)
ds = ds.cast_column('audio', Audio(decode=False))

def text_of(row):
    v = row.get('text')
    return v.strip() if isinstance(v, str) and v.strip() else None

# Pass 1: buffer clips, tally per-speaker duration
buf = []
dur = defaultdict(float)
for i, row in enumerate(ds):
    if i >= MAX_CLIPS: break
    a = row.get('audio'); t = text_of(row)
    if not a or not a.get('bytes') or not t: continue
    try:
        wav, srate = sf.read(io.BytesIO(a['bytes']), dtype='float32')
    except Exception: continue
    if wav.ndim > 1: wav = wav.mean(1)
    d = len(wav) / srate
    if d < 1.0 or d > 14: continue
    s = str(row.get('speaker_id') or 'unknown')
    buf.append((s, t, wav, srate)); dur[s] += d

top = max(dur, key=dur.get)
print('Per-speaker minutes:', {k: round(v/60,1) for k,v in sorted(dur.items(), key=lambda x:-x[1])[:12]})
print('Chosen speaker:', top, '->', round(dur[top]/60,1), 'min')

# Pass 2: write the chosen speaker's clips at 22050 Hz + metadata.csv
rows = []
for j,(s,t,wav,srate) in enumerate([b for b in buf if b[0]==top]):
    if srate != 22050: wav = librosa.resample(wav, orig_sr=srate, target_sr=22050)
    wid = f'tsn_{j:05d}'
    sf.write(f'{DATA}/wavs/{wid}.wav', wav, 22050, subtype='PCM_16')
    rows.append(f'{wid}|{t}')
open(f'{DATA}/metadata.csv','w',encoding='utf-8').write('\n'.join(rows)+'\n')
print('Wrote', len(rows), 'clips to', DATA)

## 4. Preprocess for Piper (espeak Setswana phonemes)

In [ ]:
%cd /content/piper/src/python
!python -m piper_train.preprocess \
  --language tn \
  --input-dir /content/tsn_data \
  --output-dir /content/tsn_train \
  --dataset-format ljspeech \
  --single-speaker \
  --sample-rate 22050
print('Preprocess done.')


## 5. Download a base checkpoint to fine-tune from

We start from an English medium voice (same 22050/quality) and adapt it to
Setswana — far faster and needs less data than training from scratch. Phonemes
are IPA (via espeak), so cross-lingual fine-tuning transfers well.

In [ ]:
import os
BASE = '/content/base.ckpt'
if not os.path.exists(BASE):
    !wget -q -O {BASE} 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt'
print('Base checkpoint size (MB):', round(os.path.getsize(BASE)/1e6,1))


## 6. Train

Saves checkpoints to Drive every epoch. Let it run; the longer the better.
You can stop and re-run this cell later to resume (it resumes from Drive).

In [ ]:
%cd /content/piper/src/python
import glob, os
CKPT_DIR = f'{WORK}/tsn_lightning'
os.makedirs(CKPT_DIR, exist_ok=True)
# resume from the newest Drive checkpoint if one exists, else the English base
prev = sorted(glob.glob(f'{CKPT_DIR}/**/*.ckpt', recursive=True), key=os.path.getmtime)
resume = prev[-1] if prev else '/content/base.ckpt'
print('Resuming from:', resume)
!python -m piper_train \
  --dataset-dir /content/tsn_train \
  --default_root_dir {CKPT_DIR} \
  --accelerator gpu --devices 1 \
  --batch-size 16 \
  --validation-split 0.02 --num-test-examples 2 \
  --max_epochs 6000 \
  --checkpoint-epochs 1 \
  --precision 32 \
  --resume_from_checkpoint {resume}


## 7. Export the trained voice to ONNX + test Setswana

In [ ]:
%cd /content/piper/src/python
import glob, os
last = sorted(glob.glob(f'{WORK}/tsn_lightning/**/*.ckpt', recursive=True), key=os.path.getmtime)[-1]
print('Exporting:', last)
!python -m piper_train.export_onnx {last} /content/tsn.onnx
!cp /content/tsn_train/config.json /content/tsn.onnx.json

# Synthesize a Setswana greeting and listen
!pip -q install piper-tts 2>/dev/null
text = 'Dumela, o amogetswe mo go VoiceAI. Nka go thusa jang gompieno?'
open('/content/say.txt','w',encoding='utf-8').write(text)
!cat /content/say.txt | piper -m /content/tsn.onnx -f /content/test.wav
from IPython.display import Audio
Audio('/content/test.wav')


## 8. Save the voice (small!) to Drive + download

The `.onnx` + `.onnx.json` together are ~60-80 MB. Copy to Drive (safe) and
download them. Later we can run this voice on a small CPU server for the site.

In [ ]:
import shutil
shutil.copy('/content/tsn.onnx', f'{WORK}/tsn.onnx')
shutil.copy('/content/tsn.onnx.json', f'{WORK}/tsn.onnx.json')
print('Saved to Drive:', WORK)
from google.colab import files
files.download('/content/tsn.onnx')
files.download('/content/tsn.onnx.json')
